# Abstract

The fundamental goal is to produce the most accurate forecasts for a time series at both daily and 28-day (monthly) granularity. Throughout this research, we exclusively evaluate the accuracy of these two target granularities. However, we may leverage forecasts at intermediate aggregation levels (e.g. weekly) to improve the daily and monthly predictions. This technique is known as temporal hierarchical forecast reconciliation. The objective is to explore how different intermediate aggregation levels can most effectively improve our target forecasts.

# Base hierarchies

The first big chapter looks into five so called base hierarchies. The way of the analysis is explained first.

## NRMSE Error metrics analysis

This notebook analyses the `nrmse` error metrics of the defined `Base Hierarchies` for both the base forecasts, reconciled forecasts, and their difference. The workflow was the following:
1. Choosing dataset and filtering out a subset of it for faster computation

The dataset chosen for this was a Wikipedia Web traffic dataset consisting of roughly 145k independent time series, with about two years of daily granularity history. Out of the 145k time series, I filtered for the ones, which did not have any missing data, which 130k time series satisfied. Out of these, I selected the first 10 percent (13k) for feasible computations. I expect the findings to not change significantly by having additional forecasts computated for more time series.

2. Defining 5 distinct hierarchies

The five hierarchies defined were the following: `Minimal`, `Semantic`, `Minimal+Weeks`, `Weekends+Weekdays`, `Weeks+Biweeks`. These will be described in the next section.

3. Train and validation set

I defined the forecast horizon (hence the validation set size) to be 12 weeks, to keep a balance between enough monthly forecasts, and still feasibly accurate daily predictions. Hence I split the dataset at the 12 weeks from the end, and had the first half as train set, and the second half as validation set. Note that I defined a month as 28 days for convenience in this research.

4. Base Forecasts

I used my small framework to make base predictions for each aggregation level in each hierarchy. The forecast model used for these independent base forecasts is AutoETS.

5. Reconciliation

I used shrinkage based minT reconciliation, in all hierarchies to have the base forecasts reconciled, independently per hierarchy. The reason I chose shrinkage method, is because this dataset did not provide sufficient residuals for more sophisticated reconciliation methods, so this was the most advanced one, which I also read to perform quite well in research papers.

6. Error calculation

I then used my small framework to calculate 4 distinct error metrics of these base and reconciled forecasts, for each hierarchy, and both daily and monthly granularity. I then saved these error metrics, and discarded the actual forecasts, as they are not needed.

7. Error Analysis

Out of the 4 different error metrics, I will first analyse the normalized root mean square error metrics in this notebook.

## Assumptions, abilities and limitations

Throughout the project, the biggest limitation set, is that a month is defined as 28 days exactly. This decision was done to avoid unnecessary coding complexity concerning varying months lengths, which would not even yield any practical usefulness to the research.
s
### Python framework

The Python framework used in this research consists of 7 modules:
- `aggregation.py` — Defines temporal aggregation levels (Days, Weeks, Weekdays, Weekends, Fortnights, Months) and generates summing matrices. Each aggregation knows how to resample a wide-format time series dataset to its level. Any kind of additional arbitrary aggregation can be defined with its tools, as long as its interval fits in a 28 day window.
- `temporal_hierarchy.py` — Defines a `TemporalHierarchy` as an ordered list of aggregation levels. Handles resampling, base forecasting (via AutoETS through StatsForecast), storing reconciled forecasts, and producing the full summing matrix. Predefined hierarchies include `Minimal`, `Semantic`, and `Binary`. Any kind of arbitrary hierarchies may be defined by determining the aggregation levels.
- `time_series.py` — A thin wrapper around wide-format DataFrames, tracking the aggregation level and providing a flattened view for reconciliation tools.
- `forecast_reconciliation.py` — Wraps the `forecopy` (foreco) library to perform cross-sectional minT reconciliation on the flattened base forecasts and residuals. Also computes base forecast inconsistency.
- `error_metrics.py` — Computes NRMSE, MASE, WAPE, and 1-R² per time series at any granularity. Provides composable wrappers: `FilteredErrorMetrics`, `MergedErrorMetrics`, `ErrorDifference`. `MetricsIndex` enables concise and powerful multi-level indexing into the metrics table. This basically is an abstraction of a pandas table, to avoid dealing with it directly.
- `plots.py` — Plotly-based visualization functions: error histograms, QQ plots, error difference scatter plots, error-vs-error scatter/heatmaps, box/strip plots, error sign heatmaps, and error-over-inconsistency plots.
- `utils.py` — Dataset loaders (Wikipedia Web Traffic, M4, M5, Corporación Favorita), wide/long format conversions, series alignment/trimming to 28-day multiples, and dataset reduction helpers.

Overall the framework is built to be very readable and concise, with low amounts of boilerplate, so hopefully everything will be quite clear. Nevertheless, the writings will always describe in detail what is done in the code. 

## Dictionary
Some useful terms to list in advance, for easier comprehension

| Term | Definition |
| --- | --- |
| Daily/Monthly granularity | The temporal resolution/aggregation level at which errors are evaluated. Daily refers to per-day forecast errors, monthly refers to per-28-day-period forecast errors. |
| Base forecast | The independent forecast produced by AutoETS at a given aggregation level, before any reconciliation is applied. |
| Reconciled forecast | The forecast obtained after applying minT shrinkage reconciliation within a given hierarchy, ensuring coherence across aggregation levels. |
| Base forecast error / Reconciled forecast error | The NRMSE error of a base or reconciled forecast, computed per time series at a given granularity. Each time series yields one error value. |
| Error difference (vs. base) | The difference: reconciled forecast error minus base forecast error, for a given hierarchy and granularity. A positive value means the error increased (worsened) after reconciliation; a negative value means it decreased (improved). |
| Positive / Negative error difference | The error difference filtered for only positive values (error increases) or only negative values (error improvements), used to analyze the two directions separately. |
| Hierarchy difference | The difference: `Semantic` reconciled error minus `Minimal` reconciled error, per time series. A negative value means `Semantic` achieved a lower error. Used in the Semantic vs. Minimal section. |
| Error sign distribution | A joint distribution showing the signs (+/−) of error differences at daily and monthly granularity simultaneously, yielding four categories: (+,+), (−,−), (+,−), (−,+). |
| Outlier | A forecast error in the top 5% of values (per error row). Outlier analyses use log-transformed values to make extreme values comparable. |
| Extreme outlier | A very rare, very large outlier that distorts mean and variance statistics. |
| Base inconsistency | The difference between the monthly base forecast and the sum of the corresponding daily base forecasts. Measures how incoherent the base forecasts are before reconciliation. |
| QQ plot | A quantile-quantile plot comparing two error distributions. Points on the identity line mean the distributions match; deviations show where one distribution has systematically larger or smaller values. |
| Percentile / Quantile (e.g. p75, p90) | The value below which a given percentage of error values fall. Used throughout to compare distributions without sensitivity to outliers. |

## The hierarchies

Here I will define and describe the 5 dinstinct hierarchies chosen, and show their aggregation matrices for clarity. The names used here will be used throughout the notebook.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root


import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
wth = TemporalHierarchy([MONTHS, WEEKS, DAYS], 'Minimal+Weeks')
wweth = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS], 'Weekends+Weekdays')
wfth = TemporalHierarchy([MONTHS, FORTNIGHTS, WEEKS, DAYS], 'Weeks+Biweeks')

### Minimal
This hierarchy does not contain any aggregation levels other than daily and monthly forecasts, hence it's Minimal

In [ ]:
plots.plot_summing_matrix(mth.make_summing_matrix())

### Minimal+Weeks
This hierarchy contains weekly aggregations on top of daily and monthly

In [ ]:
plots.plot_summing_matrix(wth.make_summing_matrix())

### Weekends+Weekdays
This hierarchy builds on top of the last one, adding the weekdays and weekends aggregations to the hierarchy.

In [ ]:
plots.plot_summing_matrix(wweth.make_summing_matrix())

### Weeks+Biweeks
This hierarchy still builds on Minimal+Weeks, by adding a biweekly/fortnights aggregation level.

In [ ]:
plots.plot_summing_matrix(wfth.make_summing_matrix())

### Semantic
This hierarchy consists of the most levels of aggregation, and is a union of all other. On top of daily and monthly forecasts, it contains: weekdays, weekends, weeks, fortnights/biweeks. The name came from the idea that this hierarchy contains all aggregation levels which make semantic sense for why those intervals are aggregated.

In [ ]:
plots.plot_summing_matrix(sth.make_summing_matrix())

## The dataset
Url: https://www.kaggle.com/competitions/web-traffic-time-series-forecasting/data

This dataset was chosen for the following reasons:
- Daily granularity: this was of course a prerequisite for this project.
- Has exact date timestamp: some datasets are only a sequence of values, not labeled by actual timestamps. This is convenient because we want to aggregate by weekdays and weekends too.
- High variance in data: unlike commercial datasets, this dataset rarely has long intervals of constant zero values.
- Big volume of time series: although for this analysis we are only using 10% of available data, it's still useful to be able to extend this any time.

The time series in the dataset are all trimmed to start on a monday, and to have an amount of days which is a multiple of 28, as this is required my small framework. This already makes the time series end on a sunday.